# Tugas Besar 2 IF3170 Inteligensi Artifisial

## Pendahuluan

Pada tugas besar ini, kita memprediksi apakah pemesanan kamar hotel akan dibatalkan. Kita membandingkan C4.5, SVM, regresi logistik, Naive Bayes, dan KNN. Metrik utama untuk memilih kandidat adalah F1 makro, sesuai metrik Kaggle.

## Impor dan konfigurasi <a id="inisialisasi"></a>

Jalankan sel dari atas ke bawah dengan kernel `scikit-learn==1.7.1` (lihat `README.md`). Notebook berjalan di CPU dan menyimpan hasil ke `output/`.

Alur notebook: eksplorasi data, pembagian data, rekayasa fitur, pelatihan dan perbandingan per algoritma, lalu submisi.

In [ ]:
import pickle
import random
import sys
import time
from pathlib import Path
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Video, display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import chi2_contingency
from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score, f1_score,
    log_loss, precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

if sklearn_version != '1.7.1':
    raise RuntimeError(
        f'Kernel memakai scikit-learn {sklearn_version}; diperlukan versi 1.7.1. '
        'Ikuti pemasangan di README.md, lalu pilih ulang kernel.'
    )

_NOTEBOOK_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / 'src' / 'models' / 'svm').is_dir()
)
if str(_NOTEBOOK_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_NOTEBOOK_ROOT / 'src'))

from models import (
    KNNClassifier, LogisticRegressionClassifier, NaiveBayesClassifier, SVM,
    TreeAugmentedNaiveBayes, classification_cost, cost_sensitive_threshold,
    predict_cost_sensitive,
)
from models.booking_features import build_blocks, raw_groups
from models.c45 import C45DecisionTree
from models.c45.visualization import plot_tree as c45_plot_tree
from models.logreg.visualization import save_logreg_training_video
from models.naive_bayes.visualization import plot_naive_bayes_graph, plot_tan_structure
from models.svm.visualization import plot_svm_boundary

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = _NOTEBOOK_ROOT
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'output'

print(f'Root path: {ROOT}')
print(f'Data directory: {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')

## Load Data

Label `is_canceled` ada di `train.csv`. Kolom `id` dipakai untuk menyusun submisi dan tidak masuk ke model.

In [ ]:
train_file = DATA_DIR / "train.csv"
test_file = DATA_DIR / "test.csv"
train_df = pd.read_csv(train_file, dtype={'id': str})
test_df = pd.read_csv(test_file, dtype={'id': str})

## Exploratory Data Analysis <a id="eda"></a>

Kita cek distribusi target, nilai kosong, dan hubungan antarfitur pada data latih.

### 1. Gambaran Umum

In [ ]:
print("Dataset Overview:")
train_df.info()
train_df.describe(include='all')

#### Insight
- Data training berisi 27.373 booking dengan 17 fitur, terdiri atas 14 fitur numerik dan 3 fitur kategorikal. Kolom id tidak dipakai karena hanya berisi kode unik.
- Skala fitur sangat berbeda. nightly_rate_idr bernilai jutaan, sedangkan fitur lain kebanyakan di bawah 20. Model seperti KNN, SVM, dan Logistic Regression perlu scaling terlebih dahulu.

### 2. Distribusi Target

In [ ]:
target_counts = train_df['is_canceled'].value_counts()
print("Target Variable Distribution:")
print(target_counts)

#### Insight
- Sekitar 67% booking tidak dibatalkan dan 33% dibatalkan.
- Model yang selalu menebak "tidak batal" sudah mendapat accuracy 67%, jadi evaluasi perlu memakai metrik lain seperti F1 dan ROC AUC.

### 3. Missing Values

In [ ]:
missing_val_per_col = {col: train_df[col].isnull().sum() for col in train_df.columns}
for col, missing_count in missing_val_per_col.items():
    if missing_count > 0:
        print(f"Column '{col}' has {missing_count} missing values.")

#### Insight
- Delapan kolom memiliki nilai kosong, paling banyak pada num_children (11%). Kolom lain kosong kurang dari 10%.
- Sekitar 39% baris memiliki minimal satu nilai kosong, sehingga menghapus baris akan membuang terlalu banyak data. Nilai kosong lebih baik diisi dengan median untuk fitur numerik dan modus untuk fitur kategorikal.

### 4. Data duplikat dan tidak valid

In [ ]:
duplicate_rows = train_df[train_df.duplicated()]
print(f"Number of duplicate rows in the training dataset: {len(duplicate_rows)}")

In [ ]:
unused_features = ['id', 'is_canceled']

#### Insight
- Tidak ada baris duplikat. Namun, jika id diabaikan, ada 64 baris yang isinya sama persis dengan baris lain. Baris seperti ini perlu dijaga agar tidak masuk ke training dan validation sekaligus.
- Tarif 0 pada 413 booking bukan kesalahan data karena sebagian besar berasal dari kanal Complimentary.

### 5. Fitur numerik

In [ ]:
# Keluarkan ID dan target dari daftar fitur numerik.
numeric_features = train_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_features = [col for col in numeric_features if col not in unused_features]
print(f"Numeric features: {numeric_features}")


In [ ]:
# Statistik deskriptif untuk fitur numerik dan histogram distribusi
for feature in numeric_features:
    print(f"\nFeature: {feature}")
    print(train_df[feature].describe())
    plt.figure(figsize=(8, 4))
    sns.histplot(train_df[feature], kde=True, bins=30)
    plt.title(f'Distribution of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Frequency')
    plt.show()

#### Insight
- Beberapa fitur hampir selalu bernilai 0, yaitu needs_parking, is_returning_guest, prior_cancellations, dan prior_completed_bookings. Informasinya hanya ada pada sebagian kecil booking.
- days_before_arrival miring ke kanan. Sebagian besar booking dibuat paling lama sekitar 4 bulan sebelum kedatangan, tetapi ada yang sampai 448 hari.

In [ ]:
# Outlier 
for feature in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=train_df[feature])
    plt.title(f'Boxplot of {feature}')
    plt.xlabel(feature)
    plt.show()

#### Insight
- Outlier terbanyak ada pada nightly_rate_idr dan days_before_arrival. Nilainya masih wajar sehingga tidak dihapus.
- Pada fitur diskret seperti num_adults dan num_children, banyak nilai ditandai outlier hanya karena datanya didominasi satu nilai, bukan karena salah.

### 6. Fitur kategorikal

In [ ]:
categorical_features = train_df.select_dtypes(include=['object']).columns.tolist()
categorical_features = [col for col in categorical_features if col not in unused_features]
print(f"Categorical features: {categorical_features}")

In [ ]:
# Hitung jumlah booking dan tingkat pembatalan per kategori.
for feature in categorical_features:
    value_counts = train_df[feature].value_counts()
    cancel_rate = train_df.groupby(feature)['is_canceled'].mean()
    print(f"\nFeature: {feature}")
    print("Value Counts:")
    print(value_counts)
    print("Cancel Rate:")
    print(cancel_rate)

In [ ]:
for feature in categorical_features:
    rare_categories = train_df[feature].value_counts()[train_df[feature].value_counts() < 10].index.tolist()
    if rare_categories:
        print(f"\nFeature: {feature} has rare categories: {rare_categories}")


In [ ]:
for feature in categorical_features:
    train_categories = set(train_df[feature].unique())
    test_categories = set(test_df[feature].unique())
    unseen_categories = test_categories - train_categories
    if unseen_categories:
        print(f"\nFeature: {feature} has unseen categories in test data: {unseen_categories}")

#### Insight
- booking_channel paling membedakan pembatalan. Online Travel Agent paling sering batal (36%), Corporate jarang (12%), dan Complimentary tidak pernah batal.
- Pada meal_package, Half Board paling sering batal (46%). Pada room_category, Type A paling sering batal (43%).
- Full Board dan Type E muncul kurang dari 10 kali, jadi cancel rate-nya belum bisa dipercaya.

### 7. Hubungan antarfitur

In [ ]:
#Heatmap korelasi untuk fitur numerik dan target
numeric_features_with_target = numeric_features + ['is_canceled']
plt.figure(figsize=(10, 8))
sns.heatmap(train_df[numeric_features_with_target].corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

#### Insight
- days_before_arrival paling berhubungan dengan pembatalan (0,44). Makin jauh hari booking dibuat, makin besar peluangnya dibatalkan.
- special_requests berhubungan terbalik (−0,26). Tamu yang punya permintaan khusus lebih jarang membatalkan.

In [ ]:
# Hubungan antara fitur kategorikal dan target
for feature in categorical_features:
    plt.figure(figsize=(8, 4))
    sns.barplot(x=feature, y='is_canceled', data=train_df)
    plt.title(f'Cancel Rate by {feature}')
    plt.xlabel(feature)
    plt.ylabel('Cancel Rate')
    plt.xticks(rotation=45)
    plt.show()

#### Insight
- Grafik menunjukkan hal yang sama dengan tabel sebelumnya. Perbedaan paling besar ada pada booking_channel, sedangkan perbedaan antar-room_category lebih kecil.

In [ ]:
# Uji chi-square untuk tiap fitur kategorikal terhadap target.
categorical_features_with_target = categorical_features + ['is_canceled']
for feature in categorical_features:
    contingency_table = pd.crosstab(train_df[feature], train_df['is_canceled'])
    chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)
    print(f"\nChi-square test for feature: {feature}")
    print(f"Chi-square Statistic: {chi2_stat:.4f}, p-value: {p_value:.4f}, Degrees of Freedom: {dof}")
    

#### Insight
- Ketiga fitur kategorikal berhubungan dengan pembatalan (p-value < 0,0001) sehingga layak dipakai. Hubungan terkuat ada pada booking_channel.

In [ ]:
# VIF memakai diagonal pseudoinvers matriks korelasi.
corr = train_df[numeric_features].corr()
vif = np.diag(np.linalg.pinv(corr))
print("\nVariance Inflation Factor (VIF) for Numeric Features:")
for i, feature in enumerate(numeric_features):
    print(f"{feature}: {vif[i]:.2f}")

#### Insight
- Semua nilai VIF di bawah 1,6, jauh dari batas umum 5. Tidak ada fitur numerik yang saling tumpang tindih, jadi semuanya bisa dipakai.

### 8. Pola waktu

In [ ]:
# Pisahkan per tahun agar pola pembatalan bulanan bisa dibaca.
for year in train_df['arrival_year'].unique():
    for month in train_df['arrival_month'].unique():
        subset = train_df[(train_df['arrival_year'] == year) & (train_df['arrival_month'] == month)]
        if not subset.empty:
            cancel_rate = subset['is_canceled'].mean()
            print(f"Cancel rate for {year}-{month}: {cancel_rate:.2f}")

#### Insight
- Pembatalan berubah mengikuti bulan. Pada 2024, cancel rate rendah di awal dan akhir tahun (3% pada Januari, 17% pada Desember), tetapi tinggi pada Agustus–Oktober (sekitar 47%).
- Data 2023 hanya mencakup Juli–Desember sehingga tidak bisa dibandingkan langsung dengan 2024.

## Pembagian data <a id="data-split"></a>

Semua perbandingan memakai satu bagian validasi berkelompok dari `train.csv`. Baris dengan prediktor mentah yang identik tetap berada pada sisi yang sama. Bagian ini mengukur konfigurasi yang sudah ditetapkan dari percobaan pengembangan sebelumnya, bukan memilih konfigurasi baru atau mengukur skor Kaggle.

In [ ]:
FEATURE_SEED = 20261123
TARGET = 'is_canceled'
train = train_df.set_index('id', verify_integrity=True)
X = train.drop(columns=TARGET)
y = train[TARGET].to_numpy()
X_test = test_df.set_index('id', verify_integrity=True)
if X.index.hasnans or X_test.index.hasnans or not X.index.intersection(X_test.index).empty:
    raise ValueError('ID train dan test harus terisi dan saling terpisah.')
if set(X_test.columns) != set(X.columns):
    raise ValueError('Kolom prediktor train dan test berbeda.')
X_test = X_test[X.columns]

groups = raw_groups(X)
train_idx, valid_idx = next(StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=FEATURE_SEED,
).split(X, y, groups))
assert set(groups[train_idx]).isdisjoint(groups[valid_idx])
X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
y_train, y_valid = y[train_idx], y[valid_idx]
display(pd.DataFrame([
    {'bagian': name, 'baris': len(idx), 'grup': len(np.unique(groups[idx])),
     'proporsi batal': y[idx].mean()}
    for name, idx in [('latih', train_idx), ('validasi', valid_idx)]
]).round(4))

## Prapemrosesan dan rekayasa fitur <a id="preprocessing"></a>

Konfigurasi submisi di bawah diambil dari hasil pengembangan yang tercatat. Setiap variasi dibandingkan pada input dan split yang sama sejauh memungkinkan. Untuk submisi, resepnya tetap: C4.5 membagi seluruh data berlabel menjadi *grow* dan *prune*, SVM mengambil 16.000 baris, dan tiga model lain mengambil 8.000 baris secara terstratifikasi. Ukuran ini juga dipakai pada pelatihan untuk validasi.

### Kelompok fitur yang digunakan

- **Kalender dan perilaku:** siklus hari, minggu dan bulan, musim, log waktu tunggu, serta penanda permintaan khusus.
- **Rombongan, tarif, dan riwayat:** total tamu dan malam, log dan rasio tarif, serta proporsi pembatalan historis. Penanda nilai kosong melengkapi imputasi.
- **Target encoding dan tetangga:** tingkat pembatalan per kategori dan tetangga, serta jarak ke kelas. Fitur ini memakai referensi dari fold lain untuk baris latih (*out-of-fold*). Validasi dan test hanya memakai referensi latih.

`knn` pada daftar fitur berarti ringkasan tetangga, bukan model KNN kedua. Semua statistik transformasi dipelajari dari bagian latih. Untuk C4.5, blok fitur tambahan dipelajari dari bagian *grow* sebelum diterapkan ke *prune* dan validasi atau test.

In [ ]:
FEATURE_GROUPS = {
    'dtl': ('calendar', 'behavior'),
    'svm': ('calendar', 'behavior', 'knn'),
    'logreg': ('calendar', 'party', 'price', 'history', 'behavior', 'missing', 'te', 'knn'),
    'bayes': ('te', 'knn'),
    'knn': ('behavior',),
}
SVM_CAP, SVM_KERNEL_CAP, SHARED_CAP = 16_000, 1_000, 8_000
SVM_CUTOFF = -0.1875
SVM_WEIGHTS = {'days_before_arrival': 2.0, 'nightly_rate_idr': 2.0, 'special_requests': 1.5}
SVM_DISTANCE_FEATURES = {
    'knn__nearest_class0_d2', 'knn__nearest_class1_d2',
    'knn__class0_minus_class1_nearest_d2', 'knn__mean_d2_k15',
}


def sample_rows(labels, limit, seed):
    positions = np.arange(len(labels))
    if len(positions) <= limit:
        return positions
    picked, _ = train_test_split(positions, train_size=limit, stratify=labels, random_state=seed)
    return picked

### C4.5: fitur pohon dan partisi pruning

Pohon menangani kategori teks dan nilai kosong secara langsung. Tarif nol di luar kanal `Complimentary` dianggap kosong, kategori langka digabung, dan penanda nilai kosong serta ringkasan kalender, durasi, rombongan, dan tarif ditambahkan sebelum pelatihan.

In [ ]:
dtl_text_features = ['meal_package', 'room_category', 'booking_channel']


def dtl_fit_rare(raw):
    # kategori dengan kurang dari 10 baris, dipelajari dari data training saja
    return {col: raw[col].value_counts()[lambda counts: counts < 10].index.tolist() for col in dtl_text_features}


def dtl_preprocess(raw, rare):
    df = raw.copy()
    df[dtl_text_features] = df[dtl_text_features].astype(object)  # C4.5 mengenali kategorikal dari dtype object
    invalid_rate = (df['nightly_rate_idr'] == 0) & (df['booking_channel'] != 'Complimentary')
    df.loc[invalid_rate, 'nightly_rate_idr'] = np.nan
    df['num_children'] = df['num_children'].clip(upper=4)
    for col, categories in rare.items():
        df.loc[df[col].isin(categories), col] = 'Other'

    df['special_requests_missing'] = df['special_requests'].isna().astype(int)
    df['total_nights'] = df['weekend_nights'] + df['weekday_nights']
    df['total_guests'] = df['num_adults'] + df['num_children']
    df['month_index'] = (df['arrival_year'] - 2023) * 12 + df['arrival_month']
    arrival_date = pd.to_datetime(
        dict(year=df['arrival_year'], month=df['arrival_month'], day=df['arrival_day']), errors='coerce'
    )
    df['arrival_dayofweek'] = arrival_date.dt.dayofweek
    df['rate_per_person'] = df['nightly_rate_idr'] / df['total_guests'].replace(0, np.nan)
    return df



def prepare_dtl(raw, labels, query):
    rare = dtl_fit_rare(raw)
    features = dtl_preprocess(raw, rare)
    grow, prune = next(StratifiedGroupKFold(
        n_splits=5, shuffle=True, random_state=SEED,
    ).split(features, labels, raw_groups(raw)))
    query_raw = pd.concat([raw.iloc[prune], query], ignore_index=True)
    names = FEATURE_GROUPS['dtl']
    reference = raw.iloc[grow]
    blocks = build_blocks(reference, labels[grow], query_raw, set(names), seed=SEED)

    def append(base, side):
        parts = [blocks[name][side].add_prefix(f'block_{name}_') for name in names]
        return pd.concat([base.reset_index(drop=True), *parts], axis=1)

    a = append(features.iloc[grow], 0)
    b = append(dtl_preprocess(query_raw, rare), 1)
    bundle = {
        'rare_categories': rare, 'feature_groups': names, 'seed': SEED,
        'reference_train': reference, 'reference_labels': labels[grow],
    }
    return a, b.iloc[:len(prune)], b.iloc[len(prune):], labels[grow], labels[prune], bundle


dtl_grow, dtl_prune, dtl_valid, dtl_y_grow, dtl_y_prune, _ = prepare_dtl(X_train, y_train, X_valid)
(dtl_final_grow, dtl_final_prune, dtl_test, dtl_final_y_grow,
 dtl_final_y_prune, dtl_bundle) = prepare_dtl(X, y, X_test)

dtl_numeric = [col for col in dtl_grow if col not in dtl_text_features]
dtl_sk_preprocessor = ColumnTransformer([
    ('numeric', SimpleImputer(strategy='median'), dtl_numeric),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), dtl_text_features),
])
dtl_sk_grow = dtl_sk_preprocessor.fit_transform(dtl_grow)
dtl_sk_valid = dtl_sk_preprocessor.transform(dtl_valid)

### SVM: standardisasi, bobot, dan blok tambahan

Numerik diimputasi dengan median dan distandardisasi, sedangkan kategori diimputasi dengan modus dan dikodekan *one-hot*. Tiga fitur berbobot lebih besar sebelum penggabungan. Jarak kuadrat pada blok tetangga ditransformasi dengan log bertanda; semua kolom tambahan dikalikan 0,75. Percobaan kernel memakai sampel kecil yang sama untuk membatasi memori QP dan SMO.

In [ ]:
svm_numeric = X.select_dtypes(include='number').columns.tolist()
svm_categorical = [col for col in X if col not in svm_numeric]


def prepare_svm(raw, labels, query, limit):
    positions = sample_rows(labels, limit, SEED)
    reference, ref_y = raw.iloc[positions], labels[positions]
    preprocessor = ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()),
        ]), svm_numeric),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), svm_categorical),
    ])
    a = preprocessor.fit_transform(reference)
    b = preprocessor.transform(query)
    weights = {svm_numeric.index(name): weight for name, weight in SVM_WEIGHTS.items()}
    for position, weight in weights.items():
        a[:, position] *= weight
        b[:, position] *= weight
    names = FEATURE_GROUPS['svm']
    blocks = build_blocks(reference, ref_y, query, set(names), seed=SEED)

    def append(base, side):
        extra = pd.concat([blocks[group][side] for group in names], axis=1)
        cols = extra.columns.intersection(SVM_DISTANCE_FEATURES)
        distances = extra[cols].to_numpy(dtype=float)
        extra[cols] = np.sign(distances) * np.log1p(np.abs(distances))
        return np.column_stack((base, 0.75 * extra.to_numpy(dtype=float)))

    bundle = {
        'preprocessor': preprocessor, 'numeric_weights': weights, 'cutoff': SVM_CUTOFF,
        'feature_groups': names, 'reference_train': reference, 'reference_labels': ref_y,
        'seed': SEED,
    }
    return append(a, 0), append(b, 1), ref_y, a.shape[1], bundle


svm_X_train, svm_X_valid, svm_y_train, _, _ = prepare_svm(X_train, y_train, X_valid, SVM_KERNEL_CAP)
svm_train, svm_valid, svm_labels, svm_dim, _ = prepare_svm(X_train, y_train, X_valid, SVM_CAP)
svm_final_X, svm_test, svm_final_y, svm_final_dim, svm_bundle = prepare_svm(X, y, X_test, SVM_CAP)

### Representasi KNN, regresi logistik, dan Naive Bayes

Ketiga model berbagi imputasi dan *one-hot* pada blok dasar, lalu menerima blok tambahan masing-masing sesuai daftar di atas. Untuk perbandingan Naive Bayes kategorikal dan TAN, fitur kontinu juga disiapkan dalam bin kuantil dari data latih yang sama.

In [ ]:
def prepare_shared(raw, labels, query):
    positions = np.sort(sample_rows(labels, SHARED_CAP, FEATURE_SEED))
    reference, ref_y = raw.iloc[positions].copy(), labels[positions].copy()
    model_names = ('knn', 'logreg', 'bayes')
    needed = {group for model in model_names for group in FEATURE_GROUPS[model]}
    blocks = build_blocks(reference, ref_y, query, needed, seed=FEATURE_SEED)
    matrices = {}
    for name in model_names:
        names = ('base', *FEATURE_GROUPS[name])
        matrices[name] = tuple(pd.concat([
            blocks[group][side].add_prefix(f'{group}:') for group in names
        ], axis=1) for side in (0, 1))
    return matrices, reference, ref_y


shared_valid, shared_reference, shared_y = prepare_shared(X_train, y_train, X_valid)
shared_test, shared_final_reference, shared_final_y = prepare_shared(X, y, X_test)


def shared_bundle(name, model):
    return {
        'model': model, 'feature_groups': FEATURE_GROUPS[name], 'seed': FEATURE_SEED,
        'reference_train': shared_final_reference, 'reference_labels': shared_final_y,
        'feature_columns': tuple(shared_test[name][0].columns),
    }

In [ ]:
nb_text_features = ['meal_package', 'room_category', 'booking_channel']
nb_count_features = [
    'num_adults', 'num_children', 'weekend_nights', 'weekday_nights', 'needs_parking',
    'arrival_year', 'arrival_month', 'is_returning_guest', 'prior_cancellations', 'special_requests',
]
nb_continuous_features = ['days_before_arrival', 'nightly_rate_idr', 'prior_completed_bookings', 'arrival_day']


def nb_fit_preprocessing(raw, n_bins):
    # Semua yang dipelajari (kode teks, batas bin) hanya dari data training
    params = {'n_bins': n_bins, 'text_mapping': {}, 'bin_edges': {}}
    for col in nb_text_features:
        categories = sorted(raw[col].dropna().unique())
        params['text_mapping'][col] = {name: i for i, name in enumerate(categories)}
    quantiles = np.linspace(0, 1, n_bins + 1)[1:-1]
    for col in nb_continuous_features:
        params['bin_edges'][col] = np.unique(np.nanquantile(raw[col], quantiles))
    return params


def nb_transform(raw, params):
    columns = []
    for col in nb_text_features:
        # teks -> angka, kategori yang tidak pernah muncul di training jadi NaN (dilewati)
        columns.append(raw[col].map(params['text_mapping'][col]).to_numpy(dtype=float))
    for col in nb_count_features:
        columns.append(raw[col].to_numpy(dtype=float))
    for col in nb_continuous_features:
        values = raw[col].to_numpy(dtype=float)
        codes = np.digitize(values, params['bin_edges'][col]).astype(float)
        codes[np.isnan(values)] = np.nan
        columns.append(codes)
    return np.column_stack(columns)



NB_BINS, NB_ALPHA = 5, 2.0
nb_names = nb_text_features + nb_count_features + nb_continuous_features
nb_bin_params = nb_fit_preprocessing(shared_reference, NB_BINS)
nb_binned_train = np.nan_to_num(nb_transform(shared_reference, nb_bin_params), nan=-1)
nb_binned_valid = np.nan_to_num(nb_transform(X_valid, nb_bin_params), nan=-1)

# Kode rapat; kategori tak dikenal dipetakan ke modus data latih.
nb_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1, dtype=np.int64)
nb_codes_train = nb_encoder.fit_transform(nb_binned_train)
nb_codes_valid = nb_encoder.transform(nb_binned_valid)
for column in range(nb_codes_train.shape[1]):
    mode = np.bincount(nb_codes_train[:, column]).argmax()
    nb_codes_valid[nb_codes_valid[:, column] == -1, column] = mode

### Protokol evaluasi

F1 makro memberi bobot yang sama pada kelas batal dan tidak batal. Precision dan recall melaporkan kelas batal; ROC AUC memakai probabilitas atau skor keputusan. Perbandingan antaralgoritma memakai baris validasi yang sama dan melaporkan jumlah data latih karena anggaran pelatihan berbeda.

In [ ]:
def metrics(predictions, scores):
    return {
        'f1_makro': f1_score(y_valid, predictions, average='macro'),
        'akurasi': accuracy_score(y_valid, predictions),
        'precision': precision_score(y_valid, predictions, zero_division=0),
        'recall': recall_score(y_valid, predictions, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, scores),
    }


def model_result(model, features):
    probabilities = model.predict_proba(features)
    if isinstance(model, LogisticRegressionClassifier):
        # Implementasi ini memilih kelas positif tepat pada ambang 0,5.
        return model.classes_[(probabilities[:, 1] >= 0.5).astype(int)], probabilities[:, 1]
    return model.classes_[probabilities.argmax(axis=1)], probabilities[:, 1]


def cost_table(predictions, probabilities):
    fp_cost, fn_cost = 2, 1
    adjusted = predict_cost_sensitive(probabilities, fp_cost, fn_cost)
    threshold = cost_sensitive_threshold(fp_cost, fn_cost)
    return pd.DataFrame([
        {'keputusan': name, **classification_cost(y_valid, pred, fp_cost, fn_cost)}
        for name, pred in [('aturan model', predictions), (f'ambang {threshold:.3f}', adjusted)]
    ])


validation_results = []
validation_predictions = {}


def record_validation(name, configuration, n_train, predictions, scores):
    validation_results.append({
        'model': name, 'konfigurasi': configuration, 'baris latih': n_train,
        **metrics(predictions, scores),
    })
    validation_predictions[name] = predictions

## Decision Tree Learning (C4.5) <a id="c45"></a>

Reduced Error Pruning (REP) dinilai pada pohon yang sama sebelum dan sesudah dipangkas, tanpa oversampling. Pembanding scikit-learn memakai `criterion='entropy'` dan dua nilai `ccp_alpha` tetap. Ini pembanding berbasis CART, bukan implementasi C4.5 yang identik.

In [ ]:
def dtl_result(model, features):
    rows = features[model.feature_names].to_numpy(dtype=object)
    probabilities = np.array([model.predict_distribution(model.root, row) for row in rows])
    return model.classes[probabilities.argmax(axis=1)], probabilities[:, 1]


dtl_model = C45DecisionTree().fit(dtl_grow, dtl_y_grow)
dtl_rows = [{'varian': 'C4.5 tanpa pruning', 'simpul': dtl_model.count_nodes(),
             **metrics(*dtl_result(dtl_model, dtl_valid))}]
dtl_model.prune(dtl_prune, dtl_y_prune)
dtl_predictions, dtl_scores = dtl_result(dtl_model, dtl_valid)
dtl_rows.append({'varian': 'C4.5 dengan REP', 'simpul': dtl_model.count_nodes(),
                 **metrics(dtl_predictions, dtl_scores)})
for alpha in (0.0, 0.0002):
    library = DecisionTreeClassifier(criterion='entropy', ccp_alpha=alpha, random_state=SEED)
    library.fit(dtl_sk_grow, dtl_y_grow)
    dtl_rows.append({'varian': f'scikit-learn, ccp_alpha={alpha:g}',
                     'simpul': library.tree_.node_count,
                     **metrics(*model_result(library, dtl_sk_valid))})
record_validation('C4.5', 'REP tanpa oversampling', len(dtl_grow), dtl_predictions, dtl_scores)
display(pd.DataFrame(dtl_rows).round(4))

### Model C4.5 untuk submisi

Resep submisi memakai REP tanpa oversampling, dengan bagian *grow* dan *prune* yang disiapkan dari seluruh data berlabel. Model ini tidak dipakai untuk menghitung metrik validasi di atas.

In [ ]:
dtl_submission = C45DecisionTree().fit(dtl_final_grow, dtl_final_y_grow)
dtl_submission.prune(dtl_final_prune, dtl_final_y_prune)
dtl_bundle.update(model=dtl_submission, candidate='+'.join(FEATURE_GROUPS['dtl']))

### Bonus: visualisasi pohon

Graf menampilkan maksimal tiga cabang per simpul hingga kedalaman tiga. Visualisasi memakai pohon berpangkas dari perbandingan validasi.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
c45_plot_tree(dtl_model, top_n=3, max_depth=3, path=OUTPUT_DIR / 'c45_tree.png', figsize=(18, 7))
plt.show()

### Bonus: biaya kesalahan pada C4.5

Dengan biaya false positive dua kali false negative, uji ambang 2/3 pada estimasi peluang dari daun pohon. Ini simulasi biaya, bukan perubahan aturan submisi.

In [ ]:
display(cost_table(dtl_predictions, dtl_scores))

## Support Vector Machine (SVM) <a id="svm"></a>

Kernel linear, polinomial, dan Gaussian dibandingkan antara QP, SMO, dan `SVC` pada 1.000 baris yang sama. Konfigurasi submisi RBF–SMO dinilai terpisah karena memakai parameter dan anggaran data yang berbeda.

In [ ]:
svm_kernel_configs = {kernel: {'kernel': kernel} for kernel in ('linear', 'poly', 'rbf')}
SVM_MAX_ITER = 50_000
svm_shared_params = {
    'C': 1.0, 'degree': 3, 'gamma': 1 / svm_X_train.shape[1], 'coef0': 1, 'tol': 1e-3,
}
svm_custom_models, svm_rows = {}, []


def svm_margin_scores(model, features):
    return np.concatenate([
        model.decision_function(features[start:start + 256])
        for start in range(0, len(features), 256)
    ])


for kernel, config in svm_kernel_configs.items():
    for solver in ('qp', 'smo', 'scikit-learn'):
        if solver == 'scikit-learn':
            model = SVC(**svm_shared_params, **config)
        else:
            model = SVM(**svm_shared_params, **config, solver=solver,
                        max_iter=SVM_MAX_ITER, random_state=SEED)
            svm_custom_models[solver, kernel] = model
        started = time.perf_counter()
        model.fit(svm_X_train, svm_y_train)
        seconds = time.perf_counter() - started
        scores = svm_margin_scores(model, svm_X_valid)
        svm_rows.append({
            'kernel': kernel, 'implementasi': solver, 'detik latih': seconds,
            'konvergen': model.fit_status_ == 0, **metrics(scores >= 0, scores),
        })
display(pd.DataFrame(svm_rows).round(4))

### Model SVM untuk submisi

Resep dari pengembangan sebelumnya memakai RBF–SMO, `C=4`, `gamma=1.5 / jumlah_fitur_dasar`, dan ambang skor `-0.1875`, dengan 16.000 baris. Resep yang sama dilatih untuk validasi dan untuk submisi, masing-masing dari sumber data latihnya sendiri.

In [ ]:
def fit_rbf_smo(features, labels, base_dim):
    model = SVM(kernel='rbf', solver='smo', C=4.0, gamma=1.5 / base_dim,
                tol=1e-3, max_iter=SVM_MAX_ITER, random_state=SEED).fit(features, labels)
    if not model.converged_:
        raise RuntimeError('Konfigurasi RBF–SMO tidak konvergen.')
    return model


svm_model = fit_rbf_smo(svm_train, svm_labels, svm_dim)
svm_scores = svm_margin_scores(svm_model, svm_valid)
svm_predictions = (svm_scores >= SVM_CUTOFF).astype(int)
record_validation('SVM', 'RBF–SMO, C=4, ambang -0.1875', len(svm_train), svm_predictions, svm_scores)
display(pd.DataFrame([validation_results[-1]]).round(4))

svm_submission = fit_rbf_smo(svm_final_X, svm_final_y, svm_final_dim)
svm_bundle.update(model=svm_submission, candidate='+'.join(FEATURE_GROUPS['svm']))

### Bonus: batas keputusan pada dua komponen PCA

Visualisasi melatih QP dan SMO pada proyeksi dua dimensi dari sampel kernel. Daerah keputusan, margin, dan support vector terlihat untuk ketiga kernel. Model dua dimensi ini hanya untuk visualisasi.

In [ ]:
svm_pca = PCA(n_components=2, random_state=SEED)
svm_X_train_2d = svm_pca.fit_transform(svm_X_train)
svm_visual_params = {**svm_shared_params, 'gamma': 1 / 2}
svm_visual_models = {}
svm_visual_results = []
svm_plot_lower = svm_X_train_2d.min(axis=0)
svm_plot_upper = svm_X_train_2d.max(axis=0)
svm_plot_padding = 0.05 * np.maximum(svm_plot_upper - svm_plot_lower, 1)
svm_plot_bounds = np.column_stack((svm_plot_lower - svm_plot_padding,
                                   svm_plot_upper + svm_plot_padding))
svm_boundary_axes = {}
for svm_solver in ('qp', 'smo'):
    svm_figure, svm_axes = plt.subplots(1, 3, figsize=(10, 3.5), sharex=True, sharey=True)
    svm_boundary_axes[svm_solver] = svm_axes
    for svm_index, (svm_ax, (svm_name, svm_config)) in enumerate(zip(svm_axes, svm_kernel_configs.items())):
        svm_visual_model = SVM(
            **svm_visual_params, **svm_config, solver=svm_solver,
            max_iter=SVM_MAX_ITER, random_state=SEED,
        )
        svm_start = time.perf_counter()
        svm_visual_model.fit(svm_X_train_2d, svm_y_train)
        svm_fit_seconds = time.perf_counter() - svm_start
        svm_visual_models[(svm_solver, svm_name)] = svm_visual_model
        svm_visual_results.append({
            'solver': svm_solver, 'kernel': svm_name,
            'train_time': svm_fit_seconds, 'n_iter': svm_visual_model.n_iter_,
            'converged': svm_visual_model.converged_,
            'fit_status': svm_visual_model.fit_status_,
        })
        svm_iteration_status = (
            'iteration limit' if not svm_visual_model.converged_
            and svm_visual_model.n_iter_ >= SVM_MAX_ITER else
            'converged' if svm_visual_model.converged_ else 'stopped early'
        )
        plot_svm_boundary(
            svm_visual_model, svm_X_train_2d, svm_y_train, ax=svm_ax,
            bounds=svm_plot_bounds, show_regions=True, legend=False, compact=True,
            title=f'{svm_solver.upper()} | {svm_name} | {svm_iteration_status}, iter={svm_visual_model.n_iter_}',
        )
        svm_ax.set(
            xlabel=f'PC1 ({svm_pca.explained_variance_ratio_[0]:.1%})',
            ylabel=f'PC2 ({svm_pca.explained_variance_ratio_[1]:.1%})' if svm_index == 0 else '',
        )
    svm_legend_handles = [
        Line2D([], [], marker='o', linestyle='None', color='#0072B2', label='Observed 0 (not canceled)'),
        Line2D([], [], marker='^', linestyle='None', color='#D55E00', label='Observed 1 (canceled)'),
        Patch(facecolor='#DCEAF3', alpha=0.55, label='Predicted 0 (not canceled)'),
        Patch(facecolor='#F8E3DA', alpha=0.55, label='Predicted 1 (canceled)'),
        Line2D([], [], marker='o', linestyle='None', markerfacecolor='none', color='0.2', label='Support vectors'),
        Line2D([], [], color='black', linestyle='--', label='f = -1 / +1'),
        Line2D([], [], color='black', linestyle='-', label='f = 0'),
    ]
    svm_figure.legend(
        handles=svm_legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.01),
        ncol=4, fontsize='x-small', frameon=False,
    )
    svm_figure.tight_layout(rect=(0, 0.16, 1, 1))
    plt.show()
pd.DataFrame(svm_visual_results)

### Kurva precision–recall

Kurva membandingkan ketiga kernel SMO pada sampel eksperimen kernel. Titik menunjukkan keputusan pada ambang skor nol. Untuk bonus biaya, margin SVM tidak diperlakukan sebagai probabilitas.

In [ ]:
svm_pr_styles = {'linear': '-', 'poly': '--', 'rbf': ':'}
svm_pr_markers = {'linear': 'o', 'poly': '^', 'rbf': 's'}
svm_pr_figure, svm_pr_ax = plt.subplots(figsize=(6.4, 4.2))
svm_positive_prevalence = np.mean(np.asarray(y_valid) == 1)
svm_pr_ax.axhline(
    svm_positive_prevalence, color='0.45', linestyle='-.', linewidth=1,
    label=f'Proporsi positif = {svm_positive_prevalence:.3f}',
)
for svm_name in svm_kernel_configs:
    svm_pr_model = svm_custom_models[('smo', svm_name)]
    svm_pr_scores = svm_margin_scores(svm_pr_model, svm_X_valid)
    svm_pr_precision, svm_pr_recall, _ = precision_recall_curve(
        y_valid, svm_pr_scores, pos_label=1,
    )
    svm_pr_ap = average_precision_score(y_valid, svm_pr_scores, pos_label=1)
    svm_pr_line = svm_pr_ax.plot(
        svm_pr_recall, svm_pr_precision, linestyle=svm_pr_styles[svm_name],
        linewidth=1.8, label=f'{svm_name} (AP={svm_pr_ap:.3f})',
    )[0]
    svm_pr_predictions = (svm_pr_scores >= 0).astype(int)
    svm_pr_default_precision = precision_score(
        y_valid, svm_pr_predictions, pos_label=1, zero_division=0,
    )
    svm_pr_default_recall = recall_score(
        y_valid, svm_pr_predictions, pos_label=1, zero_division=0,
    )
    svm_pr_ax.scatter(
        svm_pr_default_recall, svm_pr_default_precision, s=40,
        marker=svm_pr_markers[svm_name], facecolors='none',
        edgecolors=svm_pr_line.get_color(), linewidths=1.4, zorder=4,
        label='_nolegend_',
    )
svm_pr_note = Line2D(
    [], [], marker='o', linestyle='None', color='0.25',
    label='Simbol titik: prediksi bawaan f ≥ 0',
)
svm_pr_ax.set(
    xlim=(0, 1), ylim=(0, 1.02), xlabel='Recall (kelas 1)',
    ylabel='Precision (kelas 1)', title='SMO pada validation fitur lengkap',
)
svm_pr_ax.grid(alpha=0.2)
svm_pr_ax.legend(handles=[*svm_pr_ax.get_lines(), svm_pr_note], fontsize='small')
svm_pr_figure.tight_layout()
plt.show()

## Regresi logistik <a id="logreg"></a>

Model buatan sendiri dan scikit-learn dibandingkan tanpa penalti, dengan L1, L2, dan Elastic Net pada input yang sama. L2 dengan kekuatan regularisasi 0,01 adalah konfigurasi yang ditetapkan untuk submisi.

In [ ]:
LOGREG_PARAMS = {
    'learning_rate': 0.05, 'max_iter': 1_000, 'reg_strength': 0.01,
    'tol': 1e-3, 'random_state': FEATURE_SEED,
}
logreg_X_train, logreg_X_valid = (matrix.to_numpy(dtype=float) for matrix in shared_valid['logreg'])
logreg_custom_models, logreg_rows = {}, []
with threadpool_limits(limits=1, user_api='blas'):
    for variant, penalty in [('none', None), ('l1', 'l1'), ('l2', 'l2'), ('elasticnet', 'elasticnet')]:
        custom = LogisticRegressionClassifier(penalty=penalty, **LOGREG_PARAMS)
        library = LogisticRegression(
            penalty=penalty, C=1 / (len(shared_y) * LOGREG_PARAMS['reg_strength']),
            solver='saga', max_iter=LOGREG_PARAMS['max_iter'], tol=LOGREG_PARAMS['tol'],
            random_state=FEATURE_SEED, **({'l1_ratio': 0.5} if penalty == 'elasticnet' else {}),
        )
        logreg_custom_models[variant] = custom
        for name, model in [('buatan sendiri', custom), ('scikit-learn', library)]:
            started = time.perf_counter()
            model.fit(logreg_X_train, shared_y)
            seconds = time.perf_counter() - started
            pred, scores = model_result(model, logreg_X_valid)
            steps = int(np.asarray(model.n_iter_).reshape(-1)[0])
            converged = model.converged_ if model is custom else steps < LOGREG_PARAMS['max_iter']
            logreg_rows.append({
                'regularisasi': variant, 'implementasi': name, 'detik latih': seconds,
                'iterasi': steps, 'konvergen': converged, 'log_loss': log_loss(y_valid, scores),
                **metrics(pred, scores),
            })
            if model is custom and variant == 'l2':
                logreg_predictions, logreg_scores = pred, scores
record_validation('Regresi logistik', 'L2, lambda=0.01', len(shared_y), logreg_predictions, logreg_scores)
display(pd.DataFrame(logreg_rows).round(4))

### Model regresi logistik untuk submisi

Model menerapkan konfigurasi L2 pada sampel submisi yang disiapkan di bagian prapemrosesan. Konfigurasi tidak berubah jika varian lain mendapat skor validasi lebih tinggi.

In [ ]:
logreg_final_X, logreg_test = (matrix.to_numpy(dtype=float) for matrix in shared_test['logreg'])
logreg_submission = LogisticRegressionClassifier(penalty='l2', **LOGREG_PARAMS)
logreg_submission.fit(logreg_final_X, shared_final_y)
logreg_bundle = shared_bundle('logreg', logreg_submission)

### Bonus: perbandingan pengoptimal

Gradient ascent dari perbandingan regularisasi dipakai ulang. Conjugate gradient dan L-BFGS dilatih pada input yang sama untuk kasus tanpa penalti dan L2. Grafik menunjukkan loss latih; tabel juga melaporkan konvergensi dan kualitas prediksi.

In [ ]:
logreg_optimizer_rows = []
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
with threadpool_limits(limits=1, user_api='blas'):
    for ax, (variant, penalty) in zip(axes, [('none', None), ('l2', 'l2')]):
        for solver in ('gradient_ascent', 'conjugate_gradient', 'lbfgs'):
            if solver == 'gradient_ascent':
                model = logreg_custom_models[variant]
            else:
                model = LogisticRegressionClassifier(solver=solver, penalty=penalty, **LOGREG_PARAMS)
                model.fit(logreg_X_train, shared_y)
            pred, scores = model_result(model, logreg_X_valid)
            logreg_optimizer_rows.append({
                'regularisasi': variant, 'pengoptimal': solver, 'iterasi': model.n_iter_,
                'konvergen': model.converged_, 'loss latih': model.loss_history_[-1],
                'log_loss validasi': log_loss(y_valid, scores), **metrics(pred, scores),
            })
            ax.plot(model.loss_history_, label=solver)
        ax.set(xlabel='Iterasi', ylabel='Loss latih', title=variant)
        ax.legend()
fig.tight_layout()
plt.show()
display(pd.DataFrame(logreg_optimizer_rows).round(4))

### Bonus: video kontur log-loss

Model demonstrasi satu fitur memperlihatkan lintasan bobot dan intersep pada bidang dua dimensi. Model demonstrasi tidak dipakai untuk prediksi submisi.

In [ ]:
video_feature = 'base:base__days_before_arrival'
video_X = shared_valid['logreg'][0][[video_feature]].to_numpy(dtype=float)
video_model = LogisticRegressionClassifier(
    learning_rate=0.1, max_iter=300, penalty=None, batch_size=None,
    tol=1e-6, random_state=SEED, record_history=True,
).fit(video_X, shared_y)
video_path = save_logreg_training_video(
    video_model, video_X, shared_y, OUTPUT_DIR / 'logreg_training.mp4',
    feature_name='days_before_arrival (distandardisasi)', fps=20,
)
display(Video(str(video_path), embed=True))

### Bonus: biaya kesalahan pada regresi logistik

Bandingkan biaya keputusan L2 terhadap ambang 2/3. Ambang ilustrasi tidak mengubah konfigurasi submisi.

In [ ]:
display(cost_table(logreg_predictions, logreg_scores))

## Naive Bayes <a id="naive-bayes"></a>

Uji dua nilai smoothing pada representasi kategorikal dan campuran. Pada representasi kategorikal, Naive Bayes buatan sendiri dan `CategoricalNB` menerima kode yang sama. Pada representasi campuran, fitur biner bersifat kategorikal dan fitur lain dimodelkan sebagai Gaussian; jalur ini memakai `te` dan `knn` untuk submisi.

In [ ]:
def nb_mixed_model(features, alpha):
    types = ['categorical' if len(np.unique(column)) <= 2 else 'gaussian' for column in features.T]
    return NaiveBayesClassifier(alpha=alpha, feature_types=types)


nb_train, nb_valid = (matrix.to_numpy(dtype=float) for matrix in shared_valid['bayes'])
nb_rows = []
for alpha in (1.0, NB_ALPHA):
    for name, model in (
        ('buatan sendiri', NaiveBayesClassifier(alpha=alpha, feature_types=['categorical'] * len(nb_names))),
        ('scikit-learn', CategoricalNB(alpha=alpha)),
    ):
        model.fit(nb_codes_train, shared_y)
        nb_rows.append({'representasi': f'{NB_BINS} bin', 'implementasi': name, 'alpha': alpha,
                        **metrics(*model_result(model, nb_codes_valid))})
    mixed = nb_mixed_model(nb_train, alpha).fit(nb_train, shared_y)
    pred, scores = model_result(mixed, nb_valid)
    nb_rows.append({'representasi': 'campuran', 'implementasi': 'buatan sendiri', 'alpha': alpha,
                    **metrics(pred, scores)})
    if alpha == NB_ALPHA:
        nb_predictions, nb_scores = pred, scores
record_validation('Naive Bayes', 'campuran, alpha=2', len(shared_y), nb_predictions, nb_scores)
display(pd.DataFrame(nb_rows).round(4))

### Model Naive Bayes untuk submisi

Gunakan representasi campuran dan `alpha=2` yang ditetapkan dari eksperimen pengembangan. Model dilatih pada sampel submisi tanpa membandingkan ulang kombinasi fitur.

In [ ]:
nb_final_X, nb_test = (matrix.to_numpy(dtype=float) for matrix in shared_test['bayes'])
nb_submission = nb_mixed_model(nb_final_X, NB_ALPHA).fit(nb_final_X, shared_final_y)
nb_bundle = shared_bundle('bayes', nb_submission)

### Bonus: graf Naive Bayes dan Tree-Augmented Naive Bayes (TAN)

Graf Naive Bayes memperlihatkan hubungan setiap fitur dengan kelas. TAN menambahkan sisi antarfitur; Laplace dan Dirichlet dibandingkan dengan Naive Bayes kategorikal pada data latih dan validasi yang sama. TAN tidak dipakai untuk submisi.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
plot_naive_bayes_graph(nb_codes_train, shared_y, nb_names, class_name='is_canceled', ax=ax)
plt.show()

tan_rows = [row for row in nb_rows if row['representasi'] == f'{NB_BINS} bin'
            and row['implementasi'] == 'buatan sendiri' and row['alpha'] == NB_ALPHA]
for smoothing in ('laplace', 'dirichlet'):
    tan = TreeAugmentedNaiveBayes(alpha=NB_ALPHA, smoothing=smoothing).fit(nb_codes_train, shared_y)
    tan_rows.append({'representasi': f'{NB_BINS} bin', 'implementasi': f'TAN {smoothing}',
                     'alpha': NB_ALPHA, **metrics(*model_result(tan, nb_codes_valid))})
    if smoothing == 'dirichlet':
        fig, ax = plt.subplots(figsize=(10, 10))
        plot_tan_structure(tan, nb_names, class_name='is_canceled', ax=ax, title='Struktur TAN Dirichlet')
        fig.savefig(OUTPUT_DIR / 'tan_structure.png', bbox_inches='tight')
        plt.show()
display(pd.DataFrame(tan_rows).round(4))

### Bonus: biaya kesalahan pada Naive Bayes

Bandingkan aturan model campuran dengan ambang 2/3. Probabilitas tidak dikalibrasi, sehingga total biaya empiris pada data validasi yang menentukan apakah perubahan ini membantu.

In [ ]:
display(cost_table(nb_predictions, nb_scores))

## K-Nearest Neighbors (KNN) <a id="knn"></a>

Dengan fitur yang sama, bandingkan `k=7`, `k=15`, dan `k=31` dengan bobot jarak, serta bobot seragam pada `k=15`. Kedua implementasi memakai definisi variasi yang sama. Submisi memakai `k=15` dan bobot jarak.

In [ ]:
KNN_K, KNN_WEIGHTS = 15, 'distance'
knn_train, knn_valid = (matrix.to_numpy(dtype=float) for matrix in shared_valid['knn'])
knn_rows = []
for k, weights in [(7, KNN_WEIGHTS), (KNN_K, KNN_WEIGHTS), (31, KNN_WEIGHTS), (KNN_K, 'uniform')]:
    for name, estimator in [('buatan sendiri', KNNClassifier), ('scikit-learn', KNeighborsClassifier)]:
        model = estimator(n_neighbors=k, weights=weights).fit(knn_train, shared_y)
        pred, scores = model_result(model, knn_valid)
        knn_rows.append({'k': k, 'bobot': weights, 'implementasi': name, **metrics(pred, scores)})
        if estimator is KNNClassifier and (k, weights) == (KNN_K, KNN_WEIGHTS):
            knn_predictions, knn_scores = pred, scores
record_validation('KNN', 'k=15, bobot jarak', len(shared_y), knn_predictions, knn_scores)
display(pd.DataFrame(knn_rows).round(4))

### Model KNN untuk submisi

Model menyimpan sampel submisi sebelumnya dengan bobot jarak. Fitur tambahan hanya menggunakan blok perilaku; tidak ada pemilihan kombinasi fitur atau ambang dari hasil validasi.

In [ ]:
knn_final_X, knn_test = (matrix.to_numpy(dtype=float) for matrix in shared_test['knn'])
knn_submission = KNNClassifier(n_neighbors=KNN_K, weights=KNN_WEIGHTS).fit(knn_final_X, shared_final_y)
knn_bundle = shared_bundle('knn', knn_submission)

### Bonus: biaya kesalahan pada KNN

Bandingkan keputusan dari voting berbobot jarak dengan ambang 2/3. Submisi tetap memakai aturan prediksi biasa.

In [ ]:
display(cost_table(knn_predictions, knn_scores))

## Perbandingan antaralgoritma <a id="model-comparison"></a>

Tabel membandingkan resep submisi yang telah dilatih untuk bagian validasi yang sama. Kolom baris latih menunjukkan jumlah baris yang membentuk model; C4.5 menggunakan bagian *prune* terpisah. Hitungan ini mencerminkan konfigurasi masing-masing, bukan perbandingan pada jumlah data yang identik. Tidak ada pelatihan atau pemilihan ulang di sini.

In [ ]:
display(pd.DataFrame(validation_results).round(4))
fig, axes = plt.subplots(2, 3, figsize=(11, 6), layout='constrained')
for ax, (name, predictions) in zip(axes.flat, validation_predictions.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_valid, predictions, labels=[0, 1], display_labels=['tidak batal', 'batal'],
        colorbar=False, values_format='d', ax=ax,
    )
    ax.set_title(name)
axes.flat[-1].axis('off')
plt.show()

## Submisi <a id="submission"></a>

Gunakan model yang sudah dilatih di bagian masing-masing dan representasi test yang disiapkan sejak awal. Lima pasangan CSV dan PKL ditulis ke `output/`. CSV mengikuti urutan `sample_submission.csv`; PKL menyimpan model yang menghasilkan CSV beserta konfigurasi dan referensi prapemrosesannya. Skor Kaggle dihitung di luar notebook.

In [ ]:
template = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'id': str})
submission_ids = pd.Index(template['id'])
if submission_ids.hasnans or not submission_ids.is_unique or set(submission_ids) != set(X_test.index):
    raise ValueError('ID template harus terisi, unik, dan cocok dengan ID test.')

final_exports = {
    'dtl': (dtl_bundle, dtl_test),
    'svm': (svm_bundle, svm_test),
    'logreg': (logreg_bundle, logreg_test),
    'bayes': (nb_bundle, nb_test),
    'knn': (knn_bundle, knn_test),
}
final_submissions = {}
for name, (bundle, features) in final_exports.items():
    model = bundle['model']
    if name == 'svm':
        labels = svm_margin_scores(model, features) >= bundle['cutoff']
    else:
        labels = model.predict(features)
    if np.shape(labels) != (len(X_test),) or not np.isin(labels, (0, 1)).all():
        raise ValueError(f'Prediksi {name} harus berupa satu label biner per ID test.')
    aligned = pd.Series(labels, index=X_test.index).reindex(submission_ids)
    final_submissions[name] = pd.DataFrame({'id': submission_ids, TARGET: aligned.to_numpy(dtype=int)})

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Siapkan semua berkas sebelum mengganti keluaran dari run sebelumnya.
with TemporaryDirectory(dir=OUTPUT_DIR) as staging:
    staging_path = Path(staging)
    for name, submission in final_submissions.items():
        submission.to_csv(staging_path / f'submission_{name}.csv', index=False)
        bundle = {**final_exports[name][0], 'sklearn_version': sklearn_version}
        with (staging_path / f'{name}.pkl').open('wb') as stream:
            pickle.dump(bundle, stream, protocol=pickle.HIGHEST_PROTOCOL)
    for artifact in sorted(staging_path.iterdir()):
        artifact.replace(OUTPUT_DIR / artifact.name)

display(pd.DataFrame([
    {'berkas': f'submission_{name}.csv', 'baris': len(submission),
     'proporsi batal': submission[TARGET].mean()}
    for name, submission in final_submissions.items()
]).round(4))